# Climatology Model Evaluation by Forcing Configuration

Evaluates the trained model on specific forcing configuration codes.
For each code all available replicates are used, giving a within-config variance estimate.

**Per-code plots:**
1. Truth vs mean prediction (all replicates overlaid)
2. Error profiles by latitude
3. Within-sim window variance

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm.notebook import tqdm

ML_DIR = Path("../..")
sys.path.insert(0, str(ML_DIR / "src"))

from ml.config import load as load_config
from ml.data.climatology import is_climatology_var
from ml.data.isca_segment import aggregated_read_field, read_segment
from ml.diagnostics import find_spinup_time, find_zonal_mean_convergence_time
from ml.diagnostics.enstrophy import mean_enstrophy
from ml.diagnostics.convergence import compute_time_mean_zonal_mean
from ml.diagnostics.spatial import cosine_latitude_weights
from ml.training.model import build_model

plt.rcParams["figure.dpi"] = 120

In [2]:
# --- Parameters --- edit these ---
CONFIG_PATH = ML_DIR / "configs" / "default.toml"
TRAINING_DIR = None  # None = derive from config

# Forcing configuration codes to evaluate.
# Each code maps to all its replicates in the simulations directory.
CODES = [
    "307dbd",   # lat0=0.0  amp=1e-10  widthy=12
    "44087c",   # lat0=60.0 amp=1e-10  widthy=16
]

In [16]:
cfg = load_config("../../ml/configs/default.toml")
sim_root = Path("../../output/barotropic_stirring-T85/simulations/")
training_dir = Path("../../output/barotropic_stirring-T85/training")


def code_label(code: str) -> str:
    nl_path = next(sim_root.glob(f"{code}-*/namelist.json"), None)
    if nl_path is None:
        return code
    s = json.load(open(nl_path))["stirring_nml"]
    return f"{code}  lat0={s['lat0']} amp={s['amplitude']:.2e} widthy={s['widthy']}"


def find_replicates(code: str) -> list[Path]:
    dirs = sorted(sim_root.glob(f"{code}-*"))
    assert dirs, f"no simulation directories found for code {code!r}"
    return dirs

for code in CODES:
    reps = find_replicates(code)
    print(f"{code_label(code)}  ->  {len(reps)} replicates")

307dbd  lat0=0.0 amp=1.00e-10 widthy=12.0  ->  4 replicates
44087c  lat0=60.0 amp=1.00e-10 widthy=16.0  ->  3 replicates


## Load model

In [17]:
clim_target = all(is_climatology_var(v) for v in cfg.data.y_vars)
model = build_model(
    cfg.model,
    dropout=cfg.training.regularization.dropout,
    zonal_mean=clim_target,
)
state_dict = torch.load(training_dir / "parameters.pt", map_location="cpu", weights_only=False)
state_dict.pop("_metadata", None)
model.load_state_dict(state_dict)
model.eval()

norm = torch.load(training_dir / "normalization.pt", map_location="cpu", weights_only=True)

n_params = sum(p.numel() for p in model.parameters())
print(f"architecture: {cfg.model.architecture}  params: {n_params:,}")

architecture: sfno  params: 1,100,609


## Evaluate

In [ ]:
def predict_windows(vor_windows: np.ndarray) -> np.ndarray:
    """vor_windows: (N, K, lat, lon) -> (N, lat)"""
    x = torch.from_numpy(vor_windows.astype(np.float32))
    with torch.no_grad():
        x_norm = (x - norm["x_mean"]) / norm["x_std"]
        y_norm = model(x_norm)
        y = y_norm * norm["y_std"] + norm["y_mean"]
    return y.squeeze(1).numpy()


def extract_windows(vor_full: np.ndarray, t_s: int, K: int, stride: int) -> np.ndarray:
    T = vor_full.shape[0]
    windows = [vor_full[t : t + K] for t in range(t_s, T - K + 1, stride)]
    if not windows:
        return np.empty((0, K, *vor_full.shape[1:]), dtype=vor_full.dtype)
    return np.stack(windows, axis=0)


def evaluate_sim(sim_dir: Path) -> dict | None:
    nc_files = sorted(sim_dir.glob(cfg.data.segment_pattern))
    if not nc_files:
        return None

    ds_cache: dict = {}
    try:
        lat = read_segment(nc_files[0], ds_cache)["lat"].values.astype(np.float64)
        vor_full = aggregated_read_field(nc_files, "vor", ds_cache)
        u_full = aggregated_read_field(nc_files, "ucomp", ds_cache)
    finally:
        for d in ds_cache.values():
            d.close()

    T = vor_full.shape[0]

    t_s = 0
    if cfg.data.spinup is not None:
        detected = find_spinup_time(mean_enstrophy(vor_full, lat), **cfg.data.spinup.to_kwargs())
        if detected is not None:
            t_s = detected

    t_c = t_s
    if cfg.data.convergence is not None:
        detected = find_zonal_mean_convergence_time(
            u_full, lat, t_s,
            threshold=cfg.data.convergence.threshold,
            hold=cfg.data.convergence.hold,
        )
        if detected is not None:
            t_c = detected

    truth = compute_time_mean_zonal_mean(u_full, t_c, T)

    K = cfg.data.windows.length
    windows = extract_windows(vor_full, t_s, K, cfg.data.windows.stride)
    if len(windows) == 0:
        return None

    preds = predict_windows(windows)

    return {
        "sim_dir": sim_dir,
        "lat": lat,
        "truth": truth,
        "predictions": preds,
        "mean_pred": preds.mean(axis=0),
        "t_s": t_s,
        "t_c": t_c,
    }

In [ ]:
code_results: dict[str, tuple[str, list[dict]]] = {}

for code in CODES:
    label = code_label(code)
    sim_dirs = find_replicates(code)
    results = []
    for sim_dir in tqdm(sim_dirs, desc=label):
        res = evaluate_sim(sim_dir)
        if res is not None:
            results.append(res)
    code_results[code] = (label, results)

print("done")

## Metrics

In [20]:
def rel_l2(pred, truth, lat):
    w = cosine_latitude_weights(lat)
    return float(np.sqrt(np.sum(w * (pred - truth) ** 2) / np.sum(w * truth ** 2)))


def rmse(pred, truth, lat):
    w = cosine_latitude_weights(lat)
    return float(np.sqrt(np.sum(w * (pred - truth) ** 2) / np.sum(w)))


header = f"  {'code / replicate':<38} {'t_s':>6} {'t_c':>6} {'rmse':>10} {'relL2':>10}"
print(header)
print("  " + "-" * (len(header) - 2))
for code, (label, results) in code_results.items():
    print(f"  {label}")
    all_rl = []
    for res in results:
        lat = res["lat"]
        r = rmse(res["mean_pred"], res["truth"], lat)
        rl = rel_l2(res["mean_pred"], res["truth"], lat)
        all_rl.append(rl)
        print(f"    {res['sim_dir'].name:<36} {res['t_s']:>6} {res['t_c']:>6} {r:>10.4f} {rl:>10.4f}")
    print(f"    {'mean relL2':>36} {'':>6} {'':>6} {'':>10} {np.mean(all_rl):>10.4f}")
    print()

  code / replicate                          t_s    t_c       rmse      relL2
  --------------------------------------------------------------------------
  307dbd  lat0=0.0 amp=1.00e-10 widthy=12.0
    307dbd-0                                 10    143     0.2827     0.0322
    307dbd-1                                 30    396     0.3580     0.0405
    307dbd-2                                 10    428     0.3960     0.0449
    307dbd-3                                 10    461     0.3129     0.0363
                              mean relL2                              0.0385

  44087c  lat0=60.0 amp=1.00e-10 widthy=16.0
    44087c-1                                 20    701     0.3317     0.0725
    44087c-2                                 10    712     0.3342     0.0699
    44087c-3                                 10    148     0.3584     0.0780
                              mean relL2                              0.0734



## Per-code plots

In [ ]:
def plot_code(label: str, results: list[dict]):
    n = len(results)
    fig, axes = plt.subplots(1, 3, figsize=(14, 5), constrained_layout=True)
    fig.suptitle(label, fontsize=10)
    lat = results[0]["lat"]

    pred_colors = plt.cm.tab10(np.linspace(0, 0.9, n))
    gray_levels = np.linspace(0.3, 0.65, n)
    truth_colors = [np.array([v, v, v, 1.0]) for v in gray_levels]

    ax = axes[0]
    for i, res in enumerate(results):
        ax.plot(res["truth"], lat, color=truth_colors[i], linestyle="--", linewidth=1.4,
                label=f"truth r{i}")
        ax.plot(res["mean_pred"], lat, color=pred_colors[i], linewidth=1.5,
                label=f"pred r{i}")
    ax.axvline(0.0, color="k", linewidth=0.4, alpha=0.4)
    ax.set_xlabel("u [m/s]")
    ax.set_ylabel("lat [deg]")
    ax.set_title("truth (gray dashed) vs prediction (colored)")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    for i, res in enumerate(results):
        err = res["mean_pred"] - res["truth"]
        rl = rel_l2(res["mean_pred"], res["truth"], lat)
        ax.plot(err, lat, color=pred_colors[i], linewidth=1.2, label=f"r{i}  {rl:.3f}")
    ax.axvline(0.0, color="k", linewidth=0.8, linestyle="--")
    ax.set_xlabel("error (pred - truth) [m/s]")
    ax.set_title("error per replicate")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    for i, res in enumerate(results):
        ax.plot(res["predictions"].std(axis=0), lat, color=pred_colors[i],
                linewidth=1.2, label=f"r{i}")
    ax.set_xlabel("prediction std [m/s]")
    ax.set_title("within-sim window variance")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    plt.show()


for code, (label, results) in code_results.items():
    plot_code(label, results)